# Download dos dados (Base dos Dados / BigQuery)

Notebook de apoio do projeto *Evolução da representação feminina na política brasileira*. Ele **só baixa os dados** e os salva em `data/` (parquet), a análise está em `artigo.ipynb`.

**Como usar:** rode as células em ordem. Cada download só consulta o BigQuery se o arquivo ainda não existir em `data/`. Para refazer um download, use `forcar=True`.

**Pré-requisitos:** dependências de requirements.txt instaladas e um projeto no Google Cloud. Instruções disponíveis em README.md.

## 1. Configuração

In [1]:
import json
import datetime as dt
from pathlib import Path

import pandas as pd
import basedosdados as bd

BILLING = "a1-aedv"          # ID do projeto no Google Cloud

# Este notebook fica em notebooks/; as pastas abaixo ficam na raiz do repositório
DATA = Path("../data")
QUERIES = Path("../queries")
DATA.mkdir(exist_ok=True)
QUERIES.mkdir(exist_ok=True)

## 2. Função auxiliar

`baixar(nome, query)` executa a consulta e salva o resultado em `data/<nome>.parquet`.

In [2]:
def baixar(nome, query, forcar=False):
    arq = DATA / f"{nome}.parquet"
    if arq.exists() and not forcar:
        print(f"{arq} já existe, lendo do disco (use forcar=True para baixar de novo).")
        return pd.read_parquet(arq)

    (QUERIES / f"{nome}.sql").write_text(query.strip() + "\n", encoding="utf-8")
    df = bd.read_sql(query, billing_project_id=BILLING)

    # colunas de data do BigQuery vêm como 'dbdate'; converte para datetime comum
    for c in df.columns:
        if str(df[c].dtype) == "dbdate":
            df[c] = pd.to_datetime(df[c])

    df.to_parquet(arq)

    print(f"Salvo: {arq} ({len(df):,} linhas, {df.shape[1]} colunas)")
    return df


def colunas(tabela, dataset="br_tse_eleicoes"):
    """Lista as colunas de uma tabela da Base dos Dados (consulta de metadados)."""
    q = f"""
    SELECT column_name, data_type
    FROM `basedosdados.{dataset}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{tabela}'
    ORDER BY ordinal_position
    """
    return bd.read_sql(q, billing_project_id=BILLING)

## 3. Explorar: quais tabelas e colunas existem?

In [3]:
tabelas = bd.read_sql(
    """
    SELECT table_name
    FROM `basedosdados.br_tse_eleicoes.INFORMATION_SCHEMA.TABLES`
    ORDER BY table_name
    """,
    billing_project_id=BILLING,
)
tabelas

Downloading: 100%|██████████|


,table_name
0,bens_candidato
1,candidatos
2,despesas_candidato
3,detalhes_votacao_municipio
4,detalhes_votacao_municipio_zona
5,detalhes_votacao_secao
6,dicionario
7,local_secao
8,partidos
9,perfil_eleitorado_local_votacao


In [4]:
# Descomente para ver as colunas de cada tabela de interesse
colunas("candidatos")
# colunas("resultados_candidato")
# colunas("despesas_candidato")
# colunas("receitas_candidato")

Downloading: 100%|██████████|


,column_name,data_type
0,ano,INT64
1,id_eleicao,STRING
2,tipo_eleicao,STRING
3,data_eleicao,DATE
4,sigla_uf,STRING
5,id_municipio,STRING
6,id_municipio_tse,STRING
7,titulo_eleitoral,STRING
8,cpf,STRING
9,sequencial,STRING


## 4. Candidaturas e resultados: quem foi eleito (perguntas 1, 2, 3, 4 e 5)

Inicialmente, baixamos uma tabela de candidaturas para ser usada na questão 1, mas as colunas que utilizamos na questão 1 também estavam na tabela de resultados, então decidimos usar apenas essa tabela.

In [6]:
query_resultados = (QUERIES / "resultados_candidato.sql").read_text(encoding="utf-8")
resultados = baixar("resultados_candidato", query_resultados)
resultados.shape

Downloading: 100%|██████████|
Salvo: ..\data\resultados_candidato.parquet (2,925,021 linhas, 12 colunas)


(2925021, 12)

## 5. Financiamento de campanha (pergunta 4)

A tabela `receitas_candidato` tem uma linha por receita (dezenas de milhões). A consulta em `queries/receitas_candidato.sql` já agrega no BigQuery por candidatura, fonte, origem e natureza da receita, de 2014 a 2024 (antes de 2014 os dados de financiamento são menos completos).

In [7]:
query_receitas = (QUERIES / "receitas_candidato.sql").read_text(encoding="utf-8")
receitas = baixar("receitas_candidato", query_receitas)
receitas.shape

Downloading: 100%|██████████|
Salvo: ..\data\receitas_candidato.parquet (3,189,360 linhas, 11 colunas)


(3189360, 11)

## 6. População, eleitorado e Câmara (pergunta 6)

Três fontes para comparar a composição da Câmara com a população:

- **IBGE, Censo 2022** (via Base dos Dados): população por UF e sexo, total e com 18 anos ou mais.
- **TSE** (via Base dos Dados): eleitorado por ano, UF e gênero.
- **Câmara dos Deputados** (API de dados abertos): quem exerceu mandato em cada legislatura, com sexo, UF e partido. Esta parte não usa o BigQuery: a função `baixar_camara` chama a API e salva em `data/camara_deputados.parquet`.

In [8]:
query_pop = (QUERIES / "populacao_sexo_uf.sql").read_text(encoding="utf-8")
populacao = baixar("populacao_sexo_uf", query_pop)

query_eleitorado = (QUERIES / "eleitorado_genero_uf.sql").read_text(encoding="utf-8")
eleitorado = baixar("eleitorado_genero_uf", query_eleitorado)
populacao.shape, eleitorado.shape

Downloading: 100%|██████████|
Salvo: ..\data\populacao_sexo_uf.parquet (108 linhas, 5 colunas)
Downloading: 100%|██████████|
Salvo: ..\data\eleitorado_genero_uf.parquet (912 linhas, 4 colunas)


((108, 5), (912, 4))

In [9]:
import time
import urllib.request

API_CAMARA = "https://dadosabertos.camara.leg.br/api/v2/deputados"
LEGISLATURAS = range(52, 58)   # 52 = eleita em 2002 (2003-2007) ... 57 = eleita em 2022 (2023-2027)


def _get_json(url):
    req = urllib.request.Request(url, headers={"Accept": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)


def baixar_camara(forcar=False):
    # deputados que exerceram mandato em cada legislatura, separados por sexo (a lista não traz o sexo)
    arq = DATA / "camara_deputados.parquet"
    if arq.exists() and not forcar:
        print(f"{arq} já existe, lendo do disco (use forcar=True para baixar de novo).")
        return pd.read_parquet(arq)
    partes = []
    for leg in LEGISLATURAS:
        for sexo in ["F", "M"]:
            url = f"{API_CAMARA}?idLegislatura={leg}&siglaSexo={sexo}&itens=100&ordem=ASC&ordenarPor=nome"
            while url:
                js = _get_json(url)
                d = pd.DataFrame(js["dados"])
                d["siglaSexo"] = sexo
                d["idLegislatura"] = leg
                partes.append(d)
                url = next((l["href"] for l in js.get("links", []) if l["rel"] == "next"), None)
                time.sleep(0.3)   # não sobrecarregar a API
            print(f"legislatura {leg}, sexo {sexo}: ok")
    df = pd.concat(partes, ignore_index=True)
    df = df[["id", "nome", "siglaPartido", "siglaUf", "idLegislatura", "siglaSexo"]].drop_duplicates()
    df.to_parquet(arq, index=False)
    print(f"salvo em {arq}: {len(df):,} linhas")
    return df


camara = baixar_camara()
camara.groupby(["idLegislatura", "siglaSexo"]).size().unstack()

legislatura 52, sexo F: ok
legislatura 52, sexo M: ok
legislatura 53, sexo F: ok
legislatura 53, sexo M: ok
legislatura 54, sexo F: ok
legislatura 54, sexo M: ok
legislatura 55, sexo F: ok
legislatura 55, sexo M: ok
legislatura 56, sexo F: ok
legislatura 56, sexo M: ok
legislatura 57, sexo F: ok
legislatura 57, sexo M: ok
salvo em ..\data\camara_deputados.parquet: 5,817 linhas


siglaSexo,F,M
idLegislatura,,
52,76,968
53,62,778
54,68,797
55,105,1019
56,159,910
57,151,724


## 7. Malha das UFs (pergunta 3)

Contorno dos estados, da API de malhas do IBGE, para o mapa da questão 3. Salva em `data/geo/ufs.geojson`.

In [ ]:
import requests

GEO = DATA / "geo" / "ufs.geojson"
if GEO.exists():
    print(f"{GEO} já existe.")
else:
    url = ("https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR"
           "?intrarregiao=UF&qualidade=minima&formato=application/vnd.geo+json")
    geo = requests.get(url, timeout=60).json()
    assert len(geo["features"]) == 27, "a malha deve ter as 27 UFs"
    GEO.parent.mkdir(exist_ok=True)
    GEO.write_text(json.dumps(geo), encoding="utf-8")
    print(f"salvo em {GEO}")

## 8. Checagem final

In [12]:
for arq in sorted(DATA.glob("*.parquet")):
    print(f"{arq.name:45s} {arq.stat().st_size / 1e6:8.1f} MB")

resultados["ano"].value_counts().sort_index()

camara_deputados.parquet                           0.1 MB
eleitorado_genero_uf.parquet                       0.0 MB
populacao_sexo_uf.parquet                          0.0 MB
receitas_candidato.parquet                        45.6 MB
resultados_candidato.parquet                      35.5 MB


ano
2002     17990
2004    400856
2006     19203
2008    381250
2010     22323
2012    481851
2014     26069
2016    497264
2018     29109
2020    557313
2022     28972
2024    462821
Name: count, dtype: Int64